In [7]:
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from dotenv import load_dotenv
from openai import OpenAI
import json
import os

load_dotenv(override=True)

console = Console()
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)   

### data storage 

In [8]:
habits = []
completed = []
streaks = []

### Load Saved Service

In [9]:
def load_data():
    global habits, completed, streaks
    try:
        with open("habits.json", "r") as f:
            data = json.load(f)
            habits = data["habits"]
            completed = data["completed"]
            streaks = data["streaks"]

        console.print("[green]Habits loaded successfully[/green]")
    except:
        console.print("[yellow]No saved habits found[/yellow]")

In [10]:
load_data()

No saved habits found

### Saved Data

In [11]:
def save_data():
    data = {
        "habits": habits,
        "completed": completed,
        "streaks": streaks
    }

    with open("habits.json", "w") as f:
        json.dump(data, f)

    console.print("[cyan]Habits saved[/cyan]")

### Add Habit

In [12]:
def add_habit(habit):
    habits.append(habit)
    completed.append(False)
    streaks.append(0)
    save_data()

    console.print(f"[bold green]Habit added:[/bold green] {habit}")

### complete Habit

In [13]:
def complete_habit(index):
    if 0 <= index < len(habits):
        completed[index] = True
        streaks[index] += 1
        save_data()

        console.print(f"[green]Completed:[/green] {habits[index]}")
    else:
        console.print("[red]Invalid habit number[/red]")

### Show Habit Report

In [14]:
def get_habit_report():
    table = Table(title="Habit Tracker")

    table.add_column("Habit #")
    table.add_column("Habit")
    table.add_column("Status")
    table.add_column("Streak")

    for i, habit in enumerate(habits):
        status = "✅ Done" if completed[i] else "⬜ Pending"
        table.add_row(str(i + 1), habit, status, str(streaks[i]))

    console.print(table)

In [15]:
get_habit_report()

            Habit Tracker            
┏━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━┓
┃ Habit # ┃ Habit ┃ Status ┃ Streak ┃
┡━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━┩
└─────────┴───────┴────────┴────────┘

### Ai Habit Sugestion

In [16]:
def suggest_habits(goal):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "You are a productivity coach."},
            {"role": "user", "content": f"Suggest 5 daily habits for this goal: {goal}"}
        ]
    )

    suggestions = response.choices[0].message.content
    console.print(Panel(suggestions, title="AI Habit Suggestions"))

In [17]:
suggest_habits("becoming a better programmer")

╭───────────────────────────────────────────── AI Habit Suggestions ──────────────────────────────────────────────╮
│ To become a better programmer, here are 5 daily habits you can adopt:                                           │
│                                                                                                                 │
│ 1. **Code Every Day**                                                                                           │
│    Practice coding daily, even if just for 30 minutes. Consistent practice helps reinforce concepts and improve │
│ problem-solving skills.                                                                                         │
│                                                                                                                 │
│ 2. **Read Code and Documentation**                                                                              │
│    Spend time reading other people's code, open-source projects, or official documentation. This broadens your  │
│ understanding of different coding styles and best practices.                                                    │
│                                                                                                                 │
│ 3. **Learn Something New**                                                                                      │
│    Dedicate time to learning new programming concepts, languages, frameworks, or tools via tutorials, articles, │
│ or courses to broaden your skill set.                                                                           │
│                                                                                                                 │
│ 4. **Solve Programming Challenges**                                                                             │
│    Work on coding problems from platforms like LeetCode, HackerRank, or Codewars to sharpen your algorithmic    │
│ thinking and debugging skills.                                                                                  │
│                                                                                                                 │
│ 5. **Review and Refactor Your Code**                                                                            │
│    Regularly revisit your previous code to improve readability, optimize performance, and incorporate better    │
│ design patterns.                                                                                                │
│                                                                                                                 │
│ Would you like me to help you create a daily schedule incorporating these habits?                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Reset Daily habit

In [18]:
def reset_daily():
    global completed
    completed = [False] * len(habits)
    save_data()

    console.print("[yellow]Daily habits reset[/yellow]")

### Delete Habit 

In [19]:
def delete_habit(index):
    if 0 <= index < len(habits):
        removed = habits.pop(index)
        completed.pop(index)
        streaks.pop(index)
        save_data()

        console.print(f"[red]Removed habit:[/red] {removed}")

### Core app


In [20]:
def run_app():
    while True:
        console.print("\n[bold cyan]Habit Tracker Menu[/bold cyan]")
        console.print("1 Add Habit")
        console.print("2 Complete Habit")
        console.print("3 Show Report")
        console.print("4 Reset Daily Habits")
        console.print("5 Delete Habit")
        console.print("6 AI Suggest Habits")
        console.print("7 Exit")

        choice = input("Choose: ")

        if choice == "1":
            habit = input("Habit name: ")
            add_habit(habit)

        elif choice == "2":
            index = int(input("Habit number: ")) - 1
            complete_habit(index)

        elif choice == "3":
            get_habit_report()

        elif choice == "4":
            reset_daily()

        elif choice == "5":
            index = int(input("Habit number: ")) - 1
            delete_habit(index)

        elif choice == "6":
            goal = input("Your goal: ")
            suggest_habits(goal)

        elif choice == "7":
            console.print("[bold green]Goodbye![/bold green]")
            break

In [ ]:
run_app()

Habit Tracker Menu

1 Add Habit

2 Complete Habit

3 Show Report

4 Reset Daily Habits

...

3 Show Report

4 Reset Daily Habits

5 Delete Habit

6 AI Suggest Habits

7 Exit